# Garmin data: insights and freshness

Checks how fresh your Garmin Connect data is (when your watch last synced, which metrics have arrived for which days) and gives an overview of the data itself.

**Before running:**
1. Log in once through the Streamlit app (`uv run streamlit run app.py`). The notebook reuses the saved session in `.garmin_tokens/`.
2. Select the project's `.venv` as the kernel.

In [3]:
import logging
from datetime import date, timedelta

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

from garminreader.dashboard import freshness as fr
from garminreader.dashboard import garmin_data as gd
from garminreader.dashboard import insights as ins
from garminreader.ingest.sources.garmin import auth as gc

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
# garmin_data uses st.cache_data, which warns about running outside Streamlit.
logging.getLogger("streamlit").setLevel(logging.ERROR)

N_DAYS = 30
TODAY = date.today()
START = TODAY - timedelta(days=N_DAYS - 1)

In [4]:
api = gc.try_resume_session()
if api is None:
    raise RuntimeError("No saved Garmin session. Log in once via the Streamlit app, then rerun.")
display(Markdown(f"Logged in as **{api.full_name or api.display_name}**"))

RuntimeError: No saved Garmin session. Log in once via the Streamlit app, then rerun.

## 1. Sync freshness

When Garmin last received data, read straight from the API (bypassing the app cache). A signal is flagged stale when it is older than its threshold. The latest activity has no threshold, since rest days are normal.

In [ ]:
STALE_AFTER_HOURS = {
    "device last upload": 12,
    "daily summary last sync": 12,
    "last night's sleep end": 24,
}

signals = fr.sync_signals(api, TODAY)
signals["threshold_hours"] = signals["signal"].map(STALE_AFTER_HOURS)
# A signal with a threshold but no timestamp (failed or missing) counts as stale:
# freshness that cannot be confirmed should not look fine.
signals["stale"] = signals["threshold_hours"].notna() & ~(signals["age_hours"] <= signals["threshold_hours"])
signals.style.format({"age_hours": "{:.1f}"}).apply(
    lambda row: ["background-color: #f8d7da" if row["stale"] else "" for _ in row], axis=1
)

## 2. Load daily metrics and activities

Uses the same parsing as the app. Requests that fail are listed rather than silently shown as empty days.

In [ ]:
errors = gd.FetchErrors()
daily_df = gd.build_daily_dataframe(api, api.display_name, START, TODAY, errors)
activities_df = gd.get_activities(api, api.display_name, START, TODAY, errors)
current_status = gd.get_current_status(api, api.display_name, TODAY, errors)

display(
    Markdown(
        f"Loaded **{len(daily_df)} days** and **{len(activities_df)} activities**; "
        f"**{len(errors.failures)} failed requests**."
    )
)
errors.failures

## 3. Completeness per metric

Which metrics are present for which days. Gaps on the most recent day usually mean the watch has not synced yet; gaps further back mean the watch was not worn or the feature was not available.

In [ ]:
grid = fr.coverage(daily_df)
fig = px.imshow(
    grid.T.astype(int),
    color_continuous_scale=[[0, "#e5e7eb"], [1, "#2563eb"]],
    aspect="auto",
    labels={"x": "date", "y": "metric", "color": "present"},
    title="Data present per metric and day",
)
fig.update_coloraxes(showscale=False)
fig.show()

In [ ]:
fr.metric_lag(daily_df, TODAY)

## 4. App cache policy

How the Streamlit app caches this range: *live* days are re-fetched every 10 minutes, *settled* days are cached for a week.

In [ ]:
policy = fr.cache_policy(START, TODAY)
display(policy.groupby(["tier", "ttl_minutes"]).size().rename("days").to_frame())
policy.tail(3)

## 5. Data overview

In [ ]:
KEY_METRICS = [
    "resting_hr",
    "hrv_last_night_avg",
    "sleep_score",
    "body_battery_highest",
    "avg_stress",
    "steps",
    "readiness_score",
]
daily_df[KEY_METRICS].describe().T.round(1)

In [ ]:
TREND_METRICS = ["resting_hr", "hrv_last_night_avg", "sleep_score", "body_battery_highest", "avg_stress"]

trend = daily_df.set_index("date")[TREND_METRICS]
rolling = trend.rolling(7, min_periods=3).mean().add_suffix(" (7d avg)")
long = pd.concat([trend, rolling], axis=1).reset_index().melt(id_vars="date", var_name="series", value_name="value")
long["metric"] = long["series"].str.replace(" (7d avg)", "", regex=False)
long["kind"] = long["series"].str.contains("7d avg").map({True: "7-day average", False: "daily"})

fig = px.line(
    long,
    x="date",
    y="value",
    color="kind",
    facet_row="metric",
    height=220 * len(TREND_METRICS),
    title="Recovery metrics with 7-day average",
)
fig.update_yaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [ ]:
if activities_df.empty:
    display(Markdown("No activities in this range."))
else:
    weekly = (
        activities_df.assign(week=activities_df["date"].dt.to_period("W").dt.start_time)
        .groupby(["week", "type"], as_index=False)["duration_min"]
        .sum()
    )
    px.bar(weekly, x="week", y="duration_min", color="type", title="Weekly training minutes by type").show()

## 6. Insights

The same plain-language insights the app shows, computed on this range.

In [ ]:
for insight in ins.build_all_insights(daily_df, activities_df, current_status):
    display(Markdown(f"**{insight.title}** ({insight.direction})  \n{insight.finding}"))